# Drivers 30 — Correlations
With n≈228k spatially autocorrelated transects, naive p-values are meaningless.
Everything here is an **effect size with a block-bootstrap CI**: blocks are
contiguous alongshore chunks sized from the empirical WLR correlogram
(`S1_correlogram`), with L/2 and 2L sensitivity. Spearman is primary; results
are also stratified by Typology and Region, categorical drivers get
Kruskal–Wallis ε², and driver–driver collinearity is quantified (waves,
exposure and region are strongly confounded — single-driver correlations are
descriptive, not causal).

In [1]:
import pandas as pd
from drivers.correlations import STATS_DIR

if not (STATS_DIR / "corr_numeric.csv").exists():
    from drivers.correlations import run
    run()
print(open(STATS_DIR / "block_choice.csv").read())

decorrelation_m,r1_lag10m,n_eff,block_lens_m
500.0,0.8431560332046497,19447.51602020003,"[500, 1000, 2000]"



## Headline: numeric drivers vs WLR (main block length)

In [2]:
c = pd.read_csv(STATS_DIR / "corr_numeric.csv")
main_L = c.block_len_m.median()
w = c[(c.response=="WLR") & (c.method=="spearman") & (c.block_len_m==main_L)]
w = w.sort_values("rho", key=abs, ascending=False)
w[["driver","rho","lo","hi","n","n_blocks"]].reset_index(drop=True)

,driver,rho,lo,hi,n,n_blocks
0,beach_slope_face,-0.257941,-0.295874,-0.217722,163897,2031
1,geol_age_ma,-0.134679,-0.161694,-0.107462,228538,2959
2,erodibility_ord,0.122400,0.093061,0.153732,228538,2959
3,storm_hrs_gt4m_yr,0.121128,0.086866,0.156359,228538,2959
4,wave_incidence_deg,-0.115161,-0.149205,-0.080813,228538,2959
5,tanbeta_nearshore,0.112199,0.074924,0.145618,190375,2638
6,river_supply_idx,-0.103282,-0.140100,-0.068824,228538,2959
7,form_factor,0.096676,0.064423,0.129902,228538,2959
8,hs_p99,0.095472,0.059929,0.130190,228538,2959
9,vlm_mm_yr,-0.082013,-0.116186,-0.047061,228538,2959


## Block-length sensitivity (WLR, Spearman)

In [3]:
s = c[(c.response=="WLR") & (c.method=="spearman")]
s.pivot_table(index="driver", columns="block_len_m", values="rho").round(3)

block_len_m,500,1000,2000
driver,,,
backshore_max,-0.054,-0.054,-0.054
backshore_mean,-0.052,-0.052,-0.052
beach_slope_face,-0.258,-0.258,-0.258
cge_mean,0.040,0.040,0.040
closure_depth_m,0.013,0.013,0.013
dist_M65_km,0.056,0.056,0.056
dist_mouth_km,0.027,0.027,0.027
erodibility_ord,0.122,0.122,0.122
form_factor,0.097,0.097,0.097


## Categorical drivers (Kruskal–Wallis ε², WLR)

In [4]:
pd.read_csv(STATS_DIR / "categorical_effects.csv")

,driver,response,H,eps2,k,n
0,Typology,WLR,188044.683770,0.837582,6,224509
1,Typology,SCE,55050.254891,0.245187,6,224509
2,SubTypolog,WLR,395.055347,0.002085,3,188528
3,SubTypolog,SCE,21944.338964,0.116390,3,188528
4,SHORE_TYPE,WLR,27199.563702,0.133234,20,204026
5,SHORE_TYPE,SCE,36101.039411,0.176868,20,204026
6,EXPOSURE,WLR,277.197800,0.001342,4,204389
7,EXPOSURE,SCE,3141.754859,0.015357,4,204389
8,HINTERLAND,WLR,14123.650705,0.069075,7,204389
9,HINTERLAND,SCE,12077.197300,0.059062,7,204389


## Aggregation sensitivity (transect vs 1 km vs site medians)

In [5]:
a = pd.read_csv(STATS_DIR / "aggregation_sensitivity.csv")
a.pivot_table(index="driver", columns="level", values="rho").round(3)

level,1km,site,transect
driver,,,
backshore_max,-0.071,-0.070,-0.054
backshore_mean,-0.079,-0.079,-0.052
beach_slope_face,-0.249,-0.102,-0.258
cge_mean,0.017,-0.036,0.040
closure_depth_m,0.011,-0.024,0.013
dist_M65_km,0.087,0.092,0.056
dist_mouth_km,0.032,-0.020,0.027
erodibility_ord,0.114,0.058,0.122
form_factor,0.107,0.092,0.097


## Stratified by typology (drivers act differently on sandy vs cliff coast)

In [6]:
t = pd.read_csv(STATS_DIR / "corr_numeric_by_typology.csv")
t.pivot_table(index="driver", columns="stratum", values="rho").round(3)

stratum,Accretion_HC,Accretion_LC,Erosion_HC,Erosion_LC,Unresolved
driver,,,,,
backshore_max,-0.132,-0.108,0.309,0.124,-0.022
backshore_mean,-0.115,-0.105,0.294,0.129,-0.028
beach_slope_face,-0.300,-0.225,0.028,0.117,0.001
cge_mean,0.206,0.238,0.010,-0.184,-0.064
closure_depth_m,-0.047,0.038,0.113,-0.058,-0.079
dist_M65_km,0.080,0.118,-0.145,-0.050,0.024
dist_mouth_km,-0.007,-0.031,-0.114,0.027,-0.022
erodibility_ord,0.202,0.180,-0.227,-0.139,0.020
form_factor,-0.091,-0.092,0.130,0.079,0.030
